# 01 — Pipeline de datos: extracción, carga y SQL

Este notebook muestra el flujo completo de datos del proyecto:

1. Extracción de series desde las APIs (o modo demo)
2. Carga a la base de datos relacional (SQLite)
3. Consultas SQL con JOIN, GROUP BY y funciones de ventana

> **Nota:** si aún no tienen credenciales, usen `DEMO = True` para trabajar
> con datos sintéticos mientras las tramitan.

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
from sqlalchemy import text

import config
import carga_db

# Cambiar a False cuando tengan sus credenciales en .env
DEMO = True

## 1. Extraer y cargar las series del catálogo

El catálogo de series se define en `src/config.py`. Este paso descarga todas las series y las guarda en `data/coyuntura.db`.

In [ ]:
carga_db.main(demo=DEMO)

## 2. Explorar la base de datos con SQL

La gracia de tener los datos en una base relacional es poder consolidarlos con SQL. Veamos las tablas del esquema:

In [ ]:
engine = carga_db.obtener_engine()

with engine.connect() as conn:
    tablas = pd.read_sql(text("SELECT name FROM sqlite_master WHERE type IN ('table','view')"), conn)
tablas

### Consulta con JOIN: catálogo con su clasificación temática

In [ ]:
consulta_join = text("""
SELECT s.id_serie, s.nombre, t.tema, s.fuente, s.frecuencia, s.unidad
FROM series s
JOIN temas t ON t.id_tema = s.id_tema
ORDER BY t.tema
""")
with engine.connect() as conn:
    catalogo = pd.read_sql(consulta_join, conn)
catalogo

### Consulta con GROUP BY: cobertura de cada serie

In [ ]:
consulta_groupby = text("""
SELECT s.nombre,
       MIN(o.fecha) AS primera_obs,
       MAX(o.fecha) AS ultima_obs,
       COUNT(*)     AS n_obs
FROM observaciones o
JOIN series s ON s.id_serie = o.id_serie
GROUP BY s.nombre
ORDER BY s.nombre
""")
with engine.connect() as conn:
    cobertura = pd.read_sql(consulta_groupby, conn)
cobertura

### Consulta con función de ventana: variación interanual en SQL

La transformación más usada en análisis de coyuntura, calculada directamente en la base con `LAG(..., 12)`:

In [ ]:
consulta_ventana = text("""
SELECT o.fecha, s.nombre, o.valor,
       ROUND(100.0 * (o.valor / LAG(o.valor, 12) OVER (PARTITION BY o.id_serie ORDER BY o.fecha) - 1), 2)
           AS var_interanual_pct
FROM observaciones o
JOIN series s ON s.id_serie = o.id_serie
WHERE s.frecuencia = 'M'
ORDER BY s.nombre, o.fecha
""")
with engine.connect() as conn:
    interanual = pd.read_sql(consulta_ventana, conn, parse_dates=["fecha"])
interanual.dropna().tail(8)

## 3. Siguiente paso

Con los datos en la base, el análisis continúa en `02_analisis_ejemplo.ipynb`.

**Tareas del equipo:**
- Reemplazar el catálogo de `config.py` por sus propias series (≥ 12)
- Escribir sus propias consultas en `sql/` adaptadas a su variable objetivo